# Le Petit Prince — do better language models predict the **left** hemisphere better?

Hands-on replication of the core result of
**Bonnasse-Gahot & Pallier (2024)**, *fMRI predictors based on language models of
increasing complexity recover brain left lateralization* ([arXiv:2405.17992](https://arxiv.org/abs/2405.17992)),
on the *average subject* of the Le Petit Prince fMRI corpus.

**The pipeline**

1. `Li2022PetitAverage` study → word events + BOLD, 9 runs.
2. Word-level features from a language model.
3. Convolve with the haemodynamic response and resample to the fMRI TR.
4. Ridge encoding, leave-one-run-out → one Pearson *r* per voxel.
5. Compare left vs right hemisphere.

**Before you start:** follow `README.md` (install + `download_data`). Nothing here
needs a GPU; the small models take a few minutes on CPU.

## 0. Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from lppws.study import Li2022PetitAverage, download_data
from lppws import pipeline as pl

# Where the data lives. `download_data("data")` puts it in ./data/lpp_average_subject_en
DATA_DIR = Path(os.environ.get("LPP_DIR", "data/lpp_average_subject_en"))
if not DATA_DIR.exists():
    DATA_DIR = download_data(DATA_DIR.parent)
CACHE = Path("cache")            # feature cache: recomputing embeddings is the slow part

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("data:  ", DATA_DIR.resolve())
print("device:", DEVICE)

## 1. Study → events

`Li2022PetitAverage` is a `neuralset` Study: it turns a folder on disk into a
tidy DataFrame of typed, timestamped events. Here that is ~1700 `Word` events
per run (onsets read from the TextGrid annotation) and one `Fmri` event pointing
at that run's 4D NIfTI.

In [ ]:
study = Li2022PetitAverage(path=str(DATA_DIR), query="timeline_index < 9")
events = study.run()

timelines = list(dict.fromkeys(events["timeline"]))   # 9 runs, in order
print(f"{len(events)} events across {len(timelines)} runs")
print(events["type"].value_counts().to_dict())
events[["type", "start", "duration", "text", "timeline"]].head()

## 2. The fMRI target — mask, hemispheres, mirror homologs

`load_brain` fits a nilearn masker on the study's own `mask_lpp_en.nii.gz`
(detrending, standardisation, 1/128 Hz high-pass) and works out, for every
voxel, its MNI *x* coordinate (→ hemisphere) and the index of its **mirror
homolog** in the other hemisphere. The homolog is what makes the lateralization
measure a *paired* comparison later on.

In [ ]:
brain = pl.load_brain(next(DATA_DIR.glob("**/mask_lpp_en.nii.gz")))
print(f"voxels={brain.n_voxels}  left={brain.left.sum()}  right={brain.right.sum()}"
      f"  with a mirror homolog: {brain.valid.mean():.0%}")

# (n_TR, n_voxels) per run. ~1 min: 9 x ~65 MB of NIfTI.
bold = {tl: pl.load_bold(brain, events, tl) for tl in timelines}
{tl.split(',')[-1]: Y.shape for tl, Y in list(bold.items())[:3]}

In [ ]:
# The 7 left-hemisphere language-network ROIs shipped with the data.
rois = pl.roi_membership(next(DATA_DIR.glob("**/roi_masks")), brain)
lang_left = np.zeros(brain.n_voxels, bool)
for m in rois.values():
    lang_left |= m
print({k: int(v.sum()) for k, v in rois.items()}, "-> union:", int(lang_left.sum()), "voxels")

## 3. Predictors — word features → HRF → design matrix

`HuggingFaceText` emits one embedding per word. Because the words carry onsets,
the result is a *timed signal*, which `HrfConvolve` convolves with the SPM
haemodynamic response and resamples to the fMRI TR (0.5 Hz). `layers=2/3` picks
a layer two thirds of the way up the network.

We start **non-contextual** (`contextualized=False`): every occurrence of a word
gets the same vector. That is deliberately the weak version — we make it
contextual in section 7.

In [ ]:
hrf = pl.hf_features(events, "openai-community/gpt2",
                     contextualized=False, device=DEVICE, cache_dir=CACHE)

X = pl.design_matrix(hrf, events, timelines[0])
print("design matrix for run 1:", X.shape, " (n_TR, n_features)")
print("BOLD for run 1:        ", bold[timelines[0]].shape)

## 4. Ridge encoding, leave-one-run-out

For each of the 9 runs: fit ridge on the other 8, predict the held-out run,
correlate prediction with the real BOLD, per voxel. The 9 correlation maps are
averaged.

The ridge penalty is chosen by `RidgeCV` **inside the training runs only**.
Selecting it on the held-out run is a classic and large source of inflation.

In [ ]:
# per_fold=True keeps the 9 held-out-run maps; we need them for a real error bar.
r_static_folds = pl.encode_corr(hrf, events, timelines, bold, per_fold=True)   # ~2 min
r_static = r_static_folds.mean(0)
print(f"per-fold maps: {r_static_folds.shape}")
print(f"mean encoding r over {brain.n_voxels} voxels: {r_static.mean():.4f}")


## 5. Left vs right

Two ways to ask the question:

* **Unpaired** — mean *r* over all left voxels vs all right voxels.
* **Paired** — for each left voxel, its *r* minus the *r* of its right mirror
  homolog. This cancels any global shift in fit quality and is much more
  sensitive.

In [ ]:
L, R, li = pl.hemisphere_means(r_static, brain)
print(f"unpaired:  L={L:.4f}  R={R:.4f}   (L-R)/(|L|+|R|) = {li:+.3f}")

d_all = pl.paired_lateralization(r_static, brain)                # all left voxels
d_lang = pl.paired_lateralization(r_static, brain, lang_left)    # language ROIs only
for name, d in [("whole brain", d_all), ("language ROIs", d_lang)]:
    m, lo, hi = pl.bootstrap_ci(d)
    print(f"paired {name:14s}: {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]  (n={d.size} voxels)")

### How wrong is that confidence interval?

Very. There is only **one subject** here (49 people averaged into one), so the
bootstrap above resamples *voxels*. Neighbouring voxels in a smoothed fMRI map
are strongly correlated, so 430 voxels are nowhere near 430 independent
observations and the interval comes out far too narrow.

A better-behaved (still not perfect) error bar uses the spread **across the 9
held-out runs**: compute the lateralization separately in each fold and look at
how much it moves. The folds share 8/9 of their training data, so this is not a
clean significance test either — but it is the right order of magnitude.


In [ ]:
for name, subset in [("whole brain", None), ("language ROIs", lang_left)]:
    v = pl.across_run_ci(r_static_folds, brain, subset)
    print(f"{name:14s}: {v['mean']:+.4f}  [{v['lo']:+.4f}, {v['hi']:+.4f}]"
          f"   positive in {v['runs_positive']}/{v['n_runs']} runs")


## 6. Where in the brain?

In [ ]:
from nilearn import plotting

img = brain.masker.inverse_transform(r_static)
plotting.plot_glass_brain(img, display_mode="lyrz", colorbar=True, plot_abs=False,
                          cmap="cold_hot", title="gpt2 (static) — encoding r")
plotting.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].violinplot([r_static[brain.left], r_static[brain.right]], showmeans=True, showextrema=False)
ax[0].set_xticks([1, 2]); ax[0].set_xticklabels(["left", "right"])
ax[0].axhline(0, color="k", lw=0.6); ax[0].set_ylabel("voxel encoding r")
ax[0].set_title("Per-voxel r by hemisphere")

sel = brain.left & brain.valid
ax[1].scatter(r_static[brain.pair[sel]], r_static[sel], s=5, alpha=0.25, color="crimson")
lim = [float(min(r_static.min(), 0)), float(r_static.max())]
ax[1].plot(lim, lim, "k--", lw=0.8)
ax[1].set_xlabel("right homolog r"); ax[1].set_ylabel("left voxel r")
ax[1].set_title("above the line = left-lateralized")
plt.tight_layout(); plt.show()

In [ ]:
# Per-ROI: encoding quality, and each left ROI against its right-hemisphere homolog.
names = list(rois)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(names, [r_static[m].mean() for m in rois.values()])
ax[0].set_ylabel("mean encoding r"); ax[0].set_title("Left language-network ROIs")
ax[0].tick_params(axis="x", rotation=30)

ax[1].plot(names, [pl.paired_lateralization(r_static, brain, m).mean() for m in rois.values()], "o-")
ax[1].axhline(0, color="k", lw=0.6)
ax[1].set_ylabel("r(left) - r(right homolog)"); ax[1].set_title("ROI lateralization")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 7. Does the asymmetry grow with model size?

This is the paper's claim. Two changes make the test sensitive:

1. **Contextual embeddings** — each word is embedded *inside* its running
   context, so the features carry sentence-level information rather than a
   lexicon lookup.
2. **A sweep** over Pythia model sizes (14m → 410m), all at the same final
   checkpoint.

The first run downloads several checkpoints and recomputes embeddings — minutes
and a few GB, then cached in `CACHE`. 410m is the largest that comfortably fits
8 GB of GPU memory.

In [ ]:
# Give every Word its running context (the last 32 words, ending with itself).
ctx_events = pl.add_running_context(events, timelines, n_words=32)
print(repr(ctx_events.query("type=='Word'")["context"].iloc[20]))

In [ ]:
import gc, pandas as pd

SIZES = {
    "14m":  "EleutherAI/pythia-14m",
    "70m":  "EleutherAI/pythia-70m",
    "160m": "EleutherAI/pythia-160m",
    "410m": "EleutherAI/pythia-410m",
    # "1b": "EleutherAI/pythia-1b",   # needs more than 8 GB
}
NPARAMS = {"14m": 1.4e7, "70m": 7e7, "160m": 1.6e8, "410m": 4.1e8, "1b": 1e9}

folds = {}
for tag, name in SIZES.items():
    print("==", tag, flush=True)
    h = pl.hf_features(ctx_events, name, contextualized=True, device=DEVICE, cache_dir=CACHE)
    folds[tag] = pl.encode_corr(h, ctx_events, timelines, bold, per_fold=True)
    del h; gc.collect(); torch.cuda.is_available() and torch.cuda.empty_cache()

results = {t: f.mean(0) for t, f in folds.items()}
df = pd.DataFrame({t: pl.metrics(r, brain, lang_left) for t, r in results.items()}).T
# the honest error bar: spread across held-out runs
df = df.join(pd.DataFrame({t: pl.across_run_ci(f, brain, lang_left) for t, f in folds.items()}).T
             .rename(columns=lambda c: f"run_{c}"))
df

In [ ]:
xs = [NPARAMS[t] for t in df.index]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xs, df["mean_r"], "o-")
ax[0].set_xscale("log"); ax[0].set_xlabel("parameters"); ax[0].set_ylabel("mean r")
ax[0].set_title("Encoding quality vs model size")

yerr = np.array([df["run_mean"] - df["run_lo"], df["run_hi"] - df["run_mean"]])
ax[1].errorbar(xs, df["run_mean"], yerr=yerr, fmt="o-", color="crimson", capsize=3,
               label="language ROIs (across-run 95% CI)")
ax[1].plot(xs, df["LI_global"], "o--", color="gray", label="all voxels")
ax[1].set_xscale("log"); ax[1].axhline(0, color="k", lw=0.6)
ax[1].set_xlabel("parameters"); ax[1].set_ylabel("r(left) - r(right homolog)")
ax[1].set_title("Lateralization vs model size"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 8. Which layer? (do this before believing any curve)

Everything above read every model at the same *relative* depth, `layers=2/3`.
That choice is not innocent. Sweep it on a single model and compare the spread
you get to the spread the whole size sweep gave you.

In [ ]:
LAYERS = [1/3, 2/3, 1.0]
layer_rows = {}
for lay in LAYERS:
    h = pl.hf_features(ctx_events, "openai-community/gpt2", contextualized=True,
                       layers=lay, device=DEVICE, cache_dir=CACHE)
    f = pl.encode_corr(h, ctx_events, timelines, bold, per_fold=True)
    layer_rows[round(lay, 2)] = pl.metrics(f.mean(0), brain, lang_left)
    del h; gc.collect(); torch.cuda.is_available() and torch.cuda.empty_cache()

pd.DataFrame(layer_rows).T[["mean_r", "LI_global", "LI_lang"]]

On our run this moved `mean r` from 0.037 (layer 1/3) to 0.018 (layer 2/3) and
flipped the sign of the language-ROI lateralization — a bigger swing than the
entire 14m → 410m size sweep produced. See `RESULTS.md` for the numbers.

The practical consequence: a size or training-step curve read at one fixed
relative depth is confounded with where that depth lands inside each network.
Sweep layers first, or report the curve at each model's best layer.

## 9. Where to go next

* **Training, not just size.** Pythia publishes intermediate checkpoints as
  HuggingFace *revisions*. Pass one through `hf_features`:

  ```python
  from neuralset.extractors.text import HuggingFaceTextConfig
  pl.hf_features(ctx_events, "EleutherAI/pythia-160m", contextualized=True,
                 hf_config=HuggingFaceTextConfig(model_kwargs={"revision": "step512"}))
  ```

  Valid steps: `step1, step512, step2000, step8000, step32000, step143000`.
  The paper's claim is that lateralization grows along this axis too.
* **Layers.** Sweep `layers` over `np.linspace(0, 1, n)` — lateralization often
  peaks in the middle/upper-middle of the network.
* **Context length.** `n_words` in `add_running_context` trades context richness
  against compute.
* **Bigger models** (`pythia-1b`, `1.4b`, `2.8b`) on a machine with more memory.
* **A real error bar.** Rerun on the per-subject data (`Li2022Petit` from
  `neuralfetch`, OpenNeuro ds003643) and bootstrap over *subjects*.